In [ ]:
# Step 1: Install Required Libraries
!pip install -q flask-ngrok pyngrok tensorflow plotly statsmodels pmdarima dash dash-bootstrap-components

# Step 2: Import Libraries and Suppress Warnings
import warnings, os
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input as KerasInput
from tensorflow.keras.callbacks import EarlyStopping
from statsmodels.tsa.arima.model import ARIMA
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from flask import Flask
from pyngrok import ngrok
import plotly.graph_objects as go
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc

# Step 3: Load and Preprocess Data
data = pd.read_csv('Dataset Path') 

# Step 3a: Parse dates and create new features
data['date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y', errors='coerce')
data = data.dropna(subset=['date'])
data['month'] = data['date'].dt.month
data['day_of_week'] = data['date'].dt.dayofweek
data['product_id'] = data['Store'].astype(str)

# Step 3b: Rename and keep relevant columns
data.rename(columns={'Weekly_Sales': 'sales', 'Holiday_Flag': 'promotion'}, inplace=True)
data = data[['date', 'sales', 'promotion', 'month', 'day_of_week', 'product_id']].dropna()

# Step 3c: Encode categorical variables and scale features
data['product_id'] = LabelEncoder().fit_transform(data['product_id'])
scaler = MinMaxScaler()
data[['sales', 'promotion']] = scaler.fit_transform(data[['sales', 'promotion']])

# Step 4: Train-Test Split
features = ['month', 'day_of_week', 'product_id', 'promotion']
X = data[features]
y = data['sales']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Step 4a: Assign time index for evaluation
y_train.index = pd.date_range(start="2024-01-01", periods=len(y_train), freq='D')
y_test.index = pd.date_range(start="2024-04-01", periods=len(y_test), freq='D')

# Step 5: Prepare Data for LSTM
X_train_lstm = np.expand_dims(X_train.values, axis=1)
X_val_lstm = np.expand_dims(X_val.values, axis=1)
X_test_lstm = np.expand_dims(X_test.values, axis=1)

# Step 6: Build and Train LSTM Model
lstm_model = Sequential([
    KerasInput(shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    LSTM(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lstm_history = lstm_model.fit(
    X_train_lstm, y_train, validation_data=(X_val_lstm, y_val),
    epochs=50, batch_size=32, callbacks=[early_stop], verbose=0
)
y_lstm_pred = lstm_model.predict(X_test_lstm, verbose=0).flatten()

# Step 7: Train ARIMA Model
arima_model = ARIMA(y_train, order=(5, 1, 0)).fit()
y_arima_pred = arima_model.forecast(len(y_test))

# Step 8: Train Decision Tree Model
tree_model = DecisionTreeRegressor(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)
y_tree_pred = tree_model.predict(X_test)

# Step 9: Feature Importance Calculation
tree_importance = tree_model.feature_importances_
lstm_surrogate_importance = np.std(X_test.values, axis=0)
combined_importance = (tree_importance + lstm_surrogate_importance) / 2
feature_names = features

# Step 10: Model Evaluation Metrics
y_test_reset = y_test.reset_index(drop=True)
metrics = {
    "Model": ["LSTM", "ARIMA", "Decision Tree"],
    "MAE": [
        float(mean_absolute_error(y_test_reset, y_lstm_pred)),
        float(mean_absolute_error(y_test_reset, y_arima_pred)),
        float(mean_absolute_error(y_test_reset, y_tree_pred))
    ],
    "RMSE": [
        float(np.sqrt(mean_squared_error(y_test_reset, y_lstm_pred))),
        float(np.sqrt(mean_squared_error(y_test_reset, y_arima_pred))),
        float(np.sqrt(mean_squared_error(y_test_reset, y_tree_pred)))
    ]
}

# Step 11: Initialize Dash App
app = Dash(__name__, external_stylesheets=[dbc.themes.SLATE])

# Step 12: Results Summary Text
best_model = [metrics["Model"][i] for i, v in enumerate(metrics["RMSE"]) if v == min(metrics["RMSE"])]
best_model = best_model[0]
results_summary_text = f"""
Results Summary:
- The best performing model based on RMSE is {best_model}.
- LSTM shows strong adaptability to non-linear seasonal patterns.
- ARIMA provides smoother forecasts, useful for trend-level insights.
- Decision Tree highlights important features like promotions and seasonality.
- Promotions demonstrate a consistent positive effect on sales.
- Seasonal peaks in November and December align with promotional campaigns.
- The results suggest retailers can strategically use promotions to amplify peak-season demand.
"""

# Step 13: Promotional Impact Analysis
promo_impact = data.groupby('promotion')['sales'].mean().reset_index()
promo_impact['promotion'] = promo_impact['promotion'].map({0: "No Promotion", 1: "Promotion"})

monthly_impact = data.groupby(['month', 'promotion'])['sales'].mean().reset_index()
monthly_impact['promotion'] = monthly_impact['promotion'].map({0: "No Promotion", 1: "Promotion"})
month_map = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
monthly_impact['month'] = monthly_impact['month'].map(month_map)

# Step 14: Function to Generate Promotional Figures
def make_promo_figures():
    fig1 = go.Figure(data=[
        go.Bar(
            x=promo_impact['promotion'],
            y=promo_impact['sales'],
            marker=dict(color=['deepskyblue', 'orange'])
        )
    ])
    fig1.update_layout(
        title="Overall Promotional Impact on Average Sales",
        xaxis_title="Promotion Status",
        yaxis_title="Average Sales",
        template='plotly_dark'
    )

    fig2 = go.Figure()
    for promo_status in monthly_impact['promotion'].unique():
        subset = monthly_impact[monthly_impact['promotion'] == promo_status]
        fig2.add_trace(go.Scatter(
            x=subset['month'], y=subset['sales'],
            mode='lines+markers', name=promo_status
        ))
    fig2.update_layout(
        title="Monthly Promotional Impact on Average Sales",
        xaxis_title="Month (Jan–Dec)",
        yaxis_title="Average Sales",
        template='plotly_dark'
    )
    return fig1, fig2

# Step 15: Dashboard Layout
app.layout = dbc.Container([
    html.H2("Inventory Forecast Dashboard", style={'textAlign': 'center', 'color': '#FFA'}),

    dcc.Tabs([
        # Tab 1: Predictions
        dcc.Tab(label='Actual vs Predicted', children=[
            dcc.Dropdown(
                id='model-select',
                options=[{'label': m, 'value': m} for m in ['LSTM', 'ARIMA', 'Decision Tree', 'Combined']],
                value='LSTM', style={'color': '#000'}
            ),
            dcc.Graph(id='prediction-graph')
        ]),

        # Tab 2: Promotional Impact
        dcc.Tab(label='Promotional Impact', children=[
            html.Br(),
            dcc.Dropdown(
                id='promo-select',
                options=[
                    {'label': 'Overall Impact', 'value': 'overall'},
                    {'label': 'Monthly Impact', 'value': 'monthly'}
                ],
                value='overall', style={'color': '#000'}
            ),
            dcc.Graph(id='promo-graph')
        ]),

        # Tab 3: Feature Importance
        dcc.Tab(label='Feature Importance', children=[
            dcc.Graph(figure=go.Figure(
                data=[go.Bar(
                    x=[f for _, f in sorted(zip(combined_importance, feature_names), reverse=True)],
                    y=sorted(combined_importance, reverse=True),
                    marker=dict(
                        color=sorted(combined_importance, reverse=True),
                        colorscale='Viridis'
                    )
                )]
            ).update_layout(
                title="Combined Feature Importance",
                xaxis_title="Feature",
                yaxis_title="Importance Score",
                template='plotly_dark'
            ))
        ]),

        # Tab 4: Model Comparison
        dcc.Tab(label='Model Comparison', children=[
            dcc.Graph(figure=go.Figure(data=[
                go.Bar(name='MAE', x=metrics['Model'], y=metrics['MAE'], marker_color='royalblue'),
                go.Bar(name='RMSE', x=metrics['Model'], y=metrics['RMSE'], marker_color='darkorange')
            ]).update_layout(
                title="Model Performance Comparison",
                barmode='group',
                template='plotly_dark'
            ))
        ]),

        # Tab 5: Results Summary
        dcc.Tab(label='Results Summary', children=[
            html.Div([
                html.H4("Automatic Analysis & Summary"),
                html.P(results_summary_text, style={'whiteSpace': 'pre-line', 'fontSize': '16px'}),
                html.H5("Model Metrics Comparison Table"),
                dbc.Table.from_dataframe(
                    pd.DataFrame(metrics).round(4),
                    striped=True, bordered=True, hover=True, color="dark"
                ),
                html.Br(),
                html.H5("Additional Insights"),
                html.Ul([
                    html.Li("Promotions significantly increase sales, with peak effects during holiday months."),
                    html.Li("LSTM best captures complex seasonality and irregular spikes in demand."),
                    html.Li("ARIMA is more suitable for long-term smooth forecasting."),
                    html.Li("Decision Tree highlights promotions and month as leading predictors."),
                    html.Li("Retailers can optimize inventory planning by aligning promotions with high-demand months.")
                ])
            ], style={'padding': '20px'})
        ]),
    ])
], fluid=True)

# Step 16: Callbacks for Interactive Graphs
@app.callback(Output('prediction-graph', 'figure'), Input('model-select', 'value'))
def update_graph(model):
    fig = go.Figure()
    idx = slice(0, 30)
    if model == "LSTM":
        fig.add_trace(go.Scatter(y=y_test_reset[idx], mode='lines+markers', name='Actual Sales', marker=dict(color='deepskyblue')))
        fig.add_trace(go.Scatter(y=y_lstm_pred[idx], mode='lines+markers', name='LSTM Predicted', marker=dict(color='orange')))
    elif model == "ARIMA":
        fig.add_trace(go.Scatter(y=y_test_reset[idx], mode='lines+markers', name='Actual Sales', marker=dict(color='deepskyblue')))
        fig.add_trace(go.Scatter(y=y_arima_pred[idx], mode='lines+markers', name='ARIMA Predicted', marker=dict(color='green')))
    elif model == "Decision Tree":
        fig.add_trace(go.Scatter(y=y_test_reset[idx], mode='lines+markers', name='Actual Sales', marker=dict(color='deepskyblue')))
        fig.add_trace(go.Scatter(y=y_tree_pred[idx], mode='lines+markers', name='Decision Tree Predicted', marker=dict(color='red')))
    else:
        fig.add_trace(go.Scatter(y=y_test_reset[idx], mode='lines+markers', name='Actual Sales', marker=dict(color='deepskyblue')))
        fig.add_trace(go.Scatter(y=y_lstm_pred[idx], mode='lines+markers', name='LSTM Predicted', marker=dict(color='orange')))
        fig.add_trace(go.Scatter(y=y_arima_pred[idx], mode='lines+markers', name='ARIMA Predicted', marker=dict(color='green')))
        fig.add_trace(go.Scatter(y=y_tree_pred[idx], mode='lines+markers', name='Decision Tree Predicted', marker=dict(color='red')))
    fig.update_layout(title=f"{model} - Actual vs Predicted Sales", xaxis_title="Time Index", yaxis_title="Sales", hovermode='x unified', template='plotly_dark')
    return fig

@app.callback(Output('promo-graph', 'figure'), Input('promo-select', 'value'))
def update_promo_graph(selection):
    fig1, fig2 = make_promo_figures()
    return fig1 if selection == 'overall' else fig2

# Step 17: Run Dash App with Ngrok

import time, threading, socket

PORT = 8060

# Clean up old ngrok processes and free the port
!pkill -f ngrok || true
!fuser -n tcp {PORT} -k || true

def is_port_open(port):
    import socket
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

# Authenticate ngrok
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"
if NGROK_AUTH_TOKEN.strip():
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start Dash in a background thread
def run_dash():
    app.run(host="0.0.0.0", port=PORT, debug=False)

threading.Thread(target=run_dash, daemon=True).start()

# Wait until server is running
for _ in range(20):
    if is_port_open(PORT):
        break
    time.sleep(0.5)

# Open one ngrok tunnel only
public_url = ngrok.connect(PORT, "http")
print("Public Dashboard URL:", public_url.public_url)


^C


<IPython.core.display.Javascript object>

Public Dashboard URL: https://214dc2577af1.ngrok-free.app
